In [23]:
!pip install pymongo

from datetime import datetime
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from google.colab import userdata

uri = userdata.get('MONGODB_URI')

client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Kết nối MongoDB thành công!")
except Exception as e:
    print(e)

Kết nối MongoDB thành công!


In [ ]:
db = client['QL_TRUONG_HOC']

collection_sv = db['sinhvien']
collection_phim = db['phim']

print("đã tạo thành công db và collection!\n")

đã tạo thành công db và collection!



In [ ]:
tap_sv = [
    {
        "masv": "240001", "hoten": "Nguyen Van An", "dtb": 7.0, "drl": 8.0,
        "chuyennganh": {"manganh": "KHMT", "tennganh": "Khoa hoc may tinh"},
        "dchuyennganh": [
            {"tenmon": "toan", "diem": 8.5},
            {"tenmon": "tinhoc", "diem": 9.0}
        ],
        "dienthoai": ["0123456"],
        "nguoithan": [{"hoten": "Nguyen Nhan", "namsinh": 1965, "quanhe": "cha"}]
    },
    {
        "masv": "240002", "hoten": "Dam Vinh Hoa", "dtb": 8.5, "drl": 7.0,
        "chuyennganh": {"manganh": "ECOM", "tennganh": "Thuong mai dien tu"},
        "dchuyennganh": [
            {"tenmon": "toan", "diem": 6.0},
            {"tenmon": "kinhte", "diem": 9.5}
        ],
        "dienthoai": ["0987654", "0111222"],
        "nguoithan": [{"hoten": "Dam Van Chuong", "namsinh": 1970, "quanhe": "cha"}]
    }
]
collection_sv.insert_many(tap_sv)
print("thêm dữ liệu sinh viên thành công!\n")

thêm dữ liệu sinh viên thành công!



In [ ]:
tap_phim = [
    {
        "maphim": "p01", "tenphim": "Lat Mat 6", "thoigian": 120, "tuoitoithieu": 18,
        "theloai": ["hai", "hanh dong", "tam ly"],
        "daodien": "Ly Hai",
        "ctychieu": [{"macty": "CGV", "SoLuongSuat": 500}, {"macty": "Lotte", "SoLuongSuat": 300}],
        "dienvien": [{"ten": "Huy Khanh", "tuoi": 40}, {"ten": "Diep Bao Ngoc", "tuoi": 35}]
    },
    {
        "maphim": "p02", "tenphim": "Bo Gia", "thoigian": 110, "tuoitoithieu": 16,
        "theloai": ["hai", "tam ly"],
        "daodien": "Tran Thanh",
        "ctychieu": [{"macty": "CGV", "SoLuongSuat": 800}],
        "dienvien": [{"ten": "Tuan Tran", "tuoi": 29}, {"ten": "Tran Thanh", "tuoi": 36}]
    },
    {
        "maphim": "p03", "tenphim": "Hai Phuong", "thoigian": 90, "tuoitoithieu": 13,
        "theloai": ["hanh dong", "vo thuat"],
        "daodien": "Le Van Kiet",
        "ctychieu": [{"macty": "BHD", "SoLuongSuat": 200}],
        "dienvien": [{"ten": "Ngo Thanh Van", "tuoi": 45, "theloaidien": ["vo thuat", "hanh dong"]}]
    }
]
collection_phim.insert_many(tap_phim)
print("thêm dữ liệu phim thành công!\n")

thêm dữ liệu phim thành công!



In [ ]:
print("find:\n")
#$all -> tìm phim có cả hai thể loại là "hai" và "tam ly"
phim_hai_tamly = collection_phim.find(
    {
        "theloai": {
            "$all": ["hai", "tam ly"]
        }
    },
    {"tenphim": 1, "theloai": 1, "_id": 0}
)
print(list(phim_hai_tamly))

find:

[{'tenphim': 'Lat Mat 6', 'theloai': ['hai', 'hanh dong', 'tam ly']}, {'tenphim': 'Bo Gia', 'theloai': ['hai', 'tam ly']}, {'tenphim': 'Lat Mat 6', 'theloai': ['hai', 'hanh dong', 'tam ly']}, {'tenphim': 'Bo Gia', 'theloai': ['hai', 'tam ly']}]


In [ ]:
#dùng $elemMatch - tìm sinh viên có môn "toan" với điểm >= 8
sv_toan = collection_sv.find(
    {
        "dchuyennganh": {
            "$elemMatch": {
                "tenmon": "toan",
                "diem": {"$gte": 8.0}
                }
        }
    },
    # {"hoten": 1, "dchuyennganh.$": 1, "_id": 0}
     {"hoten": 1, "dchuyennganh": 1, "_id": 0}
)
print(list(sv_toan))

[{'hoten': 'Nguyen Van An', 'dchuyennganh': [{'tenmon': 'toan', 'diem': 8.5}, {'tenmon': 'tinhoc', 'diem': 9.0}]}, {'hoten': 'Nguyen Van An', 'dchuyennganh': [{'tenmon': 'toan', 'diem': 8.5}, {'tenmon': 'tinhoc', 'diem': 9.0}]}]


In [ ]:
#dùng $expr - so sánh hai field trong cùng 1 document (lấy sv có drl > dtb)
sv_drl_gt_dtb = collection_sv.find(
    {
        "$expr": {
            "$gt": ["$drl", "$dtb"]
        }
    },
    {"hoten": 1, "dtb": 1, "drl": 1, "_id": 0}
)
print(list(sv_drl_gt_dtb))

[{'hoten': 'Nguyen Van An', 'dtb': 7.0, 'drl': 8.0}, {'hoten': 'Nguyen Van An', 'dtb': 7.0, 'drl': 8.0}]


In [ ]:
#update
# dùng $addToSet - thêm sđt (không bị trùng lặp)
collection_sv.update_one(
    {
         "masv": "240001"
    },
    {
        "$addToSet": {
            "dienthoai": {
              "$each": [
                  "0123456", "0999988"
              ]
            }
        }
    }
)
print("thêm số đt thành công")

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff00000000000000f4'), 'opTime': {'ts': Timestamp(1777085219, 6), 't': 244}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1777085219, 6), 'signature': {'hash': b'T\xc7\xea$\x0f3\xa3\xfe\x0bq!`\xca6\xef\xa4\x97t\xbd\x90', 'keyId': 7571852821146894341}}, 'operationTime': Timestamp(1777085219, 6), 'updatedExisting': True}, acknowledged=True)

In [ ]:
#update số đt
# dùng $set
collection_sv.update_one(
    {
        "masv": "240001",
        "dienthoai": "0999999"   # tìm phần tử cần sửa
    },
    {
        "$set": {
            "dienthoai.$": "11111"
        }
    }
)
print("update thành công")

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff00000000000000f4'), 'opTime': {'ts': Timestamp(1777085627, 2), 't': 244}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1777085627, 2), 'signature': {'hash': b'\x01\xd6F\xcb\xc5am~\x11\xde\x03uu5k1\x05\xb1\t\x1f', 'keyId': 7571852821146894341}}, 'operationTime': Timestamp(1777085627, 2), 'updatedExisting': True}, acknowledged=True)

In [ ]:
# dùng array_filters
collection_sv.update_one(
    {"masv": "240001"},
    {
        "$set": {
            "dienthoai.$[elem]": "99999"
        }
    },
    array_filters=[{"elem": "11111"}]
)
print("update thành công")

update thành công


In [ ]:
# dùng array_filters - chỉ tăng 1.0 điểm cho môn "toan" của sv 240002
collection_sv.update_one(
    {"masv": "240002"},
    {
        "$inc": {
            "dchuyennganh.$[monToan].diem": 1.0
        }
    },
    array_filters=[{"monToan.tenmon": "toan"}]
)
print("update thành công")

update thành công


In [ ]:
#update pipeline: tính lại dtb =  dtb + 0.5  (dùng lại giá trị của chính nó)
collection_sv.update_many(
    {"masv": "240001"},
    [
        {
            "$set": {
                "dtb": {
                    "$add": ["$dtb", 0.5]
                }
            }
        }
    ]
)
print("update thành công")

WriteError: $add only supports numeric or date types, not object, full error: {'index': 0, 'code': 14, 'errmsg': '$add only supports numeric or date types, not object'}

In [ ]:
#delete
collection_phim.delete_many({
    "maphim": "p03"
})
print("đã xóa phim có mã là p03")

đã xóa phim có mã là p03


In [ ]:
#aggregation pipeline ($match, $unwind, $group, $project)
#Tính tổng số suất chiếu của từng công ty chiếu phim (CGV, Lotte...)
suatchieu = [
    {
        "$unwind": "$ctychieu" #phân rả công ty chiếu
    },
    {
        "$group": {
            "_id": "$ctychieu.macty",
            "tongsuat": {"$sum": "$ctychieu.SoLuongSuat"}
        }
    },
    {
        "$sort": {
            "tongsuat": -1  #sắp xếp giảm dần
        }
    }
]

result = collection_phim.aggregate(suatchieu)
print("tổng xuất chiếu theo từng công ty")
for i in result:
  print(f"  + {i['_id']}: {i['tongsuat']} suất")
print()

tổng xuất chiếu theo từng công ty
  + CGV: 2600 suất
  + Lotte: 600 suất



In [ ]:
#max/min với $out và $lookup
#bài toán: tìm thể loại phim xuất hiện nhiều nhất
#Đếm số phim của từng thể loại -> Xuất ra collection tạm 'bang_tam_theloai'
dem_the_loai = [
    {
        "$unwind": "$theloai"
    },
    {
        "$group": {
            "_id": "$theloai",
            "TongSoPhim": {"$sum": 1}
        }
    },
    {
        "$project": {
             "TheLoai": "$_id", "TongSoPhim": 1, "_id": 0
         }
    },
    {"$out": "bang_tam_theloai"}
]
collection_phim.aggregate(dem_the_loai)

In [ ]:
#Từ bảng tạm, tìm ra con số LỚN NHẤT -> Xuất ra collection 'bang_tam_max'
max_the_loai = [
    {
        "$group": {
            "_id": "null",
            "MaxSoPhim": {"$max": "$TongSoPhim"}
        }
    },
    {"$out": "bang_tam_max"}
]
db.bang_tam_theloai.aggregate(max_the_loai)

In [ ]:
#dùng $lookup để join 'bang_tam_theloai' với 'bang_tam_max'
#lấy ra những thể loại có TongSoPhim == MaxSoPhim
result = [
    {
        "$lookup": {
            "from": "bang_tam_max",
            "localField": "TongSoPhim",
            "foreignField": "MaxSoPhim",
            "as": "tham_chieu"
        }
    },
    {"$match": {"tham_chieu": {"$ne": []}}}, #chỉ giữ lại những dòng join thành công
    {"$project": {"TheLoai": 1, "TongSoPhim": 1, "_id": 0}}
]

final_result = db.bang_tam_theloai.aggregate(result)

print("thể loại có nhiều phim nhất")
for i in final_result:
  print(f"  + Thể loại: '{i['TheLoai'].upper()}' với {i['TongSoPhim']} bộ phim.")
print()

thể loại có nhiều phim nhất
  + Thể loại: 'HAI' với 4 bộ phim.
  + Thể loại: 'TAM LY' với 4 bộ phim.

